In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from tqdm import tqdm
from isds_tool.PS_data.zip_tools import uzip_file

In [3]:
input_dir = r'/localnvme/data/added_data/check1022/data_mseg_c6_1023'
input_image_dir = os.path.join(input_dir, 'images')
input_label_dir = os.path.join(input_dir, 'labels')
output_label_dir = os.path.join(input_dir, 'labels_refine')

ref_dir = r'/scrinvme/huilin/isds/other_data/check1022/date_copy_check_1023_refine'


abandonment_dir_high = 'abandonment_High'
abandonment_dir_medium = 'abandonment_Medium'
broken_dir_high = 'broken_High'
broken_dir_medium = 'broken_Medium'
corrosion_dir_high = 'corrosion_High'
corrosion_dir_medium = 'corrosion_Medium'
deformation_dir_high = 'deformation_High'
deformation_dir_medium = 'deformation_Medium'

In [4]:
def get_risk_refine(input_dir, input_name):
    risk_d, risk_b, risk_a, risk_c = 0, 0, 0, 0
    if os.path.exists(os.path.join(input_dir, deformation_dir_high, input_name)):
        risk_d = 2
    if os.path.exists(os.path.join(input_dir, deformation_dir_medium, input_name)):
        risk_d = 1
    if os.path.exists(os.path.join(input_dir, broken_dir_high, input_name)):
        risk_b = 2
    if os.path.exists(os.path.join(input_dir, broken_dir_medium, input_name)):
        risk_b = 1
    if os.path.exists(os.path.join(input_dir, abandonment_dir_high, input_name)):
        risk_a = 2
    if os.path.exists(os.path.join(input_dir, abandonment_dir_medium, input_name)):
        risk_a = 1
    if os.path.exists(os.path.join(input_dir, corrosion_dir_high, input_name)):
        risk_c = 2
    if os.path.exists(os.path.join(input_dir, corrosion_dir_medium, input_name)):
        risk_c = 1
    risks = [risk_d, risk_b, risk_a, risk_c]
    return risks

def risk_refine(input_image_dir, check_image_dir, input_gt_dir, output_gt_dir):
    os.makedirs(output_gt_dir, exist_ok=True)
    image_list = os.listdir(input_image_dir)

    for image_name in tqdm(image_list):
        label_name = Path(image_name).stem + '.txt'
        input_gt_path = os.path.join(input_gt_dir, label_name)
        output_gt_path = os.path.join(output_gt_dir, label_name)
        with open(input_gt_path, 'r') as fi, open(output_gt_path, 'w') as fo:
            data = fi.readlines()
            new_data = []
            for id_line, line in enumerate(data):
                parts = line.strip().split(' ')
                category = int(parts[0])
                att_len = int(parts[1])
                atts = list(map(int, parts[2:2 + att_len]))
                polygons = list(map(float, parts[2 + att_len:]))

                obj_image_name = Path(image_name).stem + f'_{id_line}' + Path(image_name).suffix
                atts = get_risk_refine(check_image_dir, obj_image_name)

                info = [category, att_len] + atts + polygons
                new_line = ' '.join(map(str, info)) +'\n'
                new_data.append(new_line)
            fo.writelines(new_data)


In [5]:
risk_refine(input_image_dir, ref_dir, input_label_dir, output_label_dir)

100%|██████████| 285/285 [00:10<00:00, 28.06it/s]
